In [53]:
!pip install -q agentpy numpy matplotlib seaborn
import agentpy as ap
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
sns.set(context="notebook", style="whitegrid")
params = {
    'steps': 600,          # duración en ticks (1 tick = 1 s)
    'green_ns': 20,        # VERDE para Norte-Sur
    'green_ew': 20,        # VERDE para Este-Oeste
    'yellow': 3,           # ÁMBAR
    'all_red': 1,          # ALL-RED (despeje)
    # Tasas Poisson de arribo (veh/s) por aproximación
    'lambda_N': 0.10, 'lambda_S': 0.10,
    'lambda_E': 0.10, 'lambda_W': 0.10,
    # Cinemática
    'v_free': 7.0,         # m/s
    'headway': 8.0,        # m separación mínima
    # Geometría (intersección centrada en 0,0)
    'L': 80.0,             # media-calzada (desde centro al extremo de dibujo)
    'w': 3.5               # ancho de carril
}
# --- Definiciones para la rotonda ---
ENTRY_NS = {'N1','S2'}   # Entradas controladas juntas en fase NS
ENTRY_EW = {'E1','W2'}   # Entradas controladas juntas en fase EW
ALL_ENTRIES = ENTRY_NS | ENTRY_EW
ALL_EXITS = {'N2','E2','S1','W1'}

# Rutas válidas por origen de ENTRADA
ROTUNDA_ROUTES = {
    'N1': ['N2', 'W1', 'S1', 'E2'],     # U a N2, o salir a W1, S1, E2
    'E1': ['N2', 'W1', 'S1'],
    'W2': ['S1', 'E2', 'N2'],
    'S2': ['S1', 'E2', 'N2', 'W1'],
}

routes = {
    'N1': ['N2', 'W1', 'S1', 'E2'],  # entrada norte principal
    'E1': ['N2', 'W1', 'S1'],        # entrada este
    'W2': ['S1', 'E2', 'N2'],        # entrada oeste
    'S2': ['S1', 'E2', 'N2', 'W1'],  # entrada sur
    # salidas (N2, E2, S1, W1) solo reciben, no generan autos
}
import agentpy as ap
import numpy as np

# ======================================================
# 1) Controlador heurístico
# ======================================================
class FourWaySignalsHeuristic:
    """Control adaptativo heurístico (ajusta según colas)."""

    def __init__(self, model, green_ns, green_ew, yellow, all_red):
        self.model = model
        self.min_g_ns, self.min_g_ew = int(green_ns), int(green_ew)
        self.y, self.ar = int(yellow), int(all_red)
        self.phase = 0  # 0 = NS, 1 = EW
        self.sub = 'G'
        self.t_in = 0

    def lights(self):
        # Todos rojos por defecto
        L = {d: 'R' for d in ['N1','S2','E1','W2','N2','S1','E2','W1']}
        if self.phase == 0:
            L['N1'] = L['S2'] = self.sub
        else:
            L['E1'] = L['W2'] = self.sub
        # salidas siempre libres
        for exit_dir in ['N2','E2','S1','W1']:
            L[exit_dir] = 'G'
        return L

    def step(self):
        # conteo de colas en el modelo
        q_ns = sum(1 for c in self.model.cars if c.origin in ['N1','S2'] and c.state == 'stop')
        q_ew = sum(1 for c in self.model.cars if c.origin in ['E1','W2'] and c.state == 'stop')

        if self.phase == 0:  # NS
            if self.sub == 'G' and self.t_in >= self.min_g_ns and q_ns == 0:
                self.sub, self.t_in = 'Y', 0
            elif self.sub == 'Y' and self.t_in >= self.y:
                self.sub, self.t_in = 'AR', 0
            elif self.sub == 'AR' and self.t_in >= self.ar:
                self.phase, self.sub, self.t_in = 1, 'G', 0
            else:
                self.t_in += 1
        else:  # EW
            if self.sub == 'G' and self.t_in >= self.min_g_ew and q_ew == 0:
                self.sub, self.t_in = 'Y', 0
            elif self.sub == 'Y' and self.t_in >= self.y:
                self.sub, self.t_in = 'AR', 0
            elif self.sub == 'AR' and self.t_in >= self.ar:
                self.phase, self.sub, self.t_in = 0, 'G', 0
            else:
                self.t_in += 1


# ======================================================
# 2) Carros
# ======================================================
class Car(ap.Agent):
    """Auto en rotonda: entra por N1/E1/W2/S2, elige destino (N2,E2,S1,W1)."""

    def setup(self, origin):
        self.origin = origin
        self.state = 'approach'  # 'approach', 'stop', 'inside', 'go', 'done'
        self.v = self.model.p.v_free

        # Elegir destino válido
        self.dest = np.random.choice(self.model.routes[self.origin])

        L, w, R = self.model.p.L, self.model.p.w, self.model.p.R
        off = w/2
        self.R = R

        # --- Coordenadas de spawn y stopline ---
        self.spawns = {
            'N1': np.array([-w/2,  L]),
            'S2': np.array([+w/2, -L]),
            'E1': np.array([ L, +w/2]),
            'W2': np.array([-L, -w/2])
        }

        self.stoplines = {
            'N1': np.array([-w/2,  R+2]),
            'S2': np.array([+w/2, -R-2]),
            'E1': np.array([ R+2, +w/2]),
            'W2': np.array([-R-2, -w/2])
        }

        # --- Entry/exit de la rotonda ---
        entry_angle = {'N1': -90, 'E1': 180, 'S2': 90, 'W2': 0}[origin]
        exit_angle = {'N2': 90, 'E2': 0, 'S1': -90, 'W1': 180}[self.dest]

        theta_in = np.deg2rad(entry_angle)
        theta_out = np.deg2rad(exit_angle)

        self.entry_point = np.array([R*np.cos(theta_in), R*np.sin(theta_in)])
        self.exit_point  = np.array([R*np.cos(theta_out), R*np.sin(theta_out)])

        # --- Meta final ---
        self.goal = {
            'N2': np.array([+off, L]),
            'E2': np.array([L, -off]),
            'S1': np.array([+off, -L]),
            'W1': np.array([-L, +off])
        }[self.dest]

        # Path: spawn → stopline → entry → exit → goal
        self.path = [self.stoplines[origin], self.entry_point, self.exit_point, self.goal]
        self.pos = self.spawns[origin].copy()
        self.current_goal = self.path.pop(0)
        self.state = 'approach'

    def dist_to(self, p):
        return np.linalg.norm(self.pos - p)

    def step(self):
        if self.state == 'done':
            return

        # --- Antes de la rotonda: obedecer semáforo ---
        if self.state in ['approach', 'stop']:
            near_stop = self.dist_to(self.stoplines[self.origin]) < 3.0
            L = self.model.ctrl.lights()
            if near_stop and L[self.origin] != 'G':
                self.state = 'stop'
                return
            else:
                self.state = 'go'

        # --- Movimiento hacia el siguiente punto del path ---
        vec = self.current_goal - self.pos
        dist = np.linalg.norm(vec)
        if dist > 0:
            dirvec = vec / dist
            step_size = min(self.v, dist)
            self.pos += dirvec * step_size

        # --- Verificar si alcanzó el punto actual ---
        if self.dist_to(self.current_goal) < 0.5:
            if self.path:
                self.current_goal = self.path.pop(0)
                # Si entra a la rotonda
                if np.array_equal(self.current_goal, self.exit_point):
                    self.state = 'inside'
                # Si sale de la rotonda
                elif self.state == 'inside':
                    self.state = 'go'
            else:
                self.state = 'done'



ROTUNDA_ROUTES = {
    'N1': ['N2', 'W1', 'S1', 'E2'],     # U a N2, o salir a W1, S1, E2
    'E1': ['N2', 'W1', 'S1'],
    'W2': ['S1', 'E2', 'N2'],
    'S2': ['S1', 'E2', 'N2', 'W1'],
}
# ======================================================
# 3) Modelo con heurística fija
# ======================================================
class FourWayModel(ap.Model):

    def setup(self):
        p = self.p

        # Controlador heurístico fijo
        self.ctrl = FourWaySignalsHeuristic(self, p.green_ns, p.green_ew, p.yellow, p.all_red)
        self.routes = ROTUNDA_ROUTES
        self.cars = ap.AgentList(self, 0, Car)
        
        self.spawn_counts = {d:0 for d in ['N1','E1','W2','S2']}

        # métricas
        self.total_delay = 0
        self.delay_counts = 0
        self.queue_lengths = {d:[] for d in ['N1','E1','W2','S2']}
        self.done_cars = 0

    def spawn_poisson(self, origin, lam):
        k = np.random.poisson(lam)
        for _ in range(k):
            self.cars.append(Car(self, origin=origin))
            self.spawn_counts[origin]+=1

    def step(self):
        # arribos
        self.spawn_poisson('N1', self.p.lambda_N)
        self.spawn_poisson('E1', self.p.lambda_E)
        self.spawn_poisson('W2', self.p.lambda_W)
        self.spawn_poisson('S2', self.p.lambda_S)

        # señales
        self.ctrl.step()

        # autos (de momento no se mueven en el espacio)
        self.cars.step()

        # limpieza
        finished = [c for c in self.cars if c.state == 'done']
        self.done_cars += len(finished)
        self.cars = ap.AgentList(self, [c for c in self.cars if c.state != 'done'], Car)

        # métricas
        for d in ['N1','E1','W2','S2']:
            q_len = sum(1 for c in self.cars if c.origin == d and c.state == 'stop')
            self.queue_lengths[d].append(q_len)

        waiting = sum(1 for c in self.cars if c.state == 'stop')
        self.total_delay += waiting
        self.delay_counts += 1

    def headway_ahead(self, me):
        """Encuentra el coche líder en el mismo carril y sentido."""
        same = [c for c in self.cars if c is not me and np.allclose(c.dir, me.dir)]
        if not same:
            return None
        ahead = []
        for c in same:
            v = c.pos - me.pos
            proj = np.dot(v, me.dir)
            if proj > 0:  # está adelante
                ahead.append((proj, c))
        if not ahead:
            return None
        return min(ahead, key=lambda x: x[0])[1]

    def results(self):
        delay_avg = self.total_delay / max(1, self.delay_counts)
        max_queues = {d: max(self.queue_lengths[d]) if self.queue_lengths[d] else 0 
                      for d in self.queue_lengths}
        return {
            "delay_avg": delay_avg,
            "throughput": self.done_cars,
            "max_queues": max_queues
        }


# ======================================================
# 4) Parámetros
# ======================================================
params_rotonda = {
    'steps': 600,
    'green_ns': 20,
    'green_ew': 20,
    'yellow': 3,
    'all_red': 1,
    'lambda_N': 0.15,   # N1
    'lambda_S': 0.10,   # S2
    'lambda_E': 0.12,   # E1
    'lambda_W': 0.08,   # W2
    'v_free': 7.0,
    'headway': 8.0,
    'L': 80.0,
    'w': 3.5,
    'R': 15.0,   # <<< nuevo radio de la rotonda
    'ctrl_class': FourWaySignalsHeuristic
}
# ======================================================
# 5) Ejecución
# ======================================================
model_rotonda = FourWayModel(params_rotonda)


def draw_intersection(ax, L, w):
    ax.clear()
    ax.set_xlim(-L, L); ax.set_ylim(-L, L)
    ax.set_aspect('equal')
    ax.set_xticks([]); ax.set_yticks([])

    # Rotonda
    circ = plt.Circle((0,0), 15, color="#d0d0d0", zorder=0)
    ax.add_patch(circ)

    # Calles (rectángulos hacia el centro)
    # Oeste
    ax.add_patch(plt.Rectangle((-L, -w), L, 2*w, color='#e0e0e0', zorder=0))
    # Norte
    ax.add_patch(plt.Rectangle((-w, 0), 2*w, L, color='#e0e0e0', zorder=0))
    # Sur
    ax.add_patch(plt.Rectangle((-w, -L), 2*w, L, color='#e0e0e0', zorder=0))
    # Este
    ax.add_patch(plt.Rectangle((0, -w), L, 2*w, color='#e0e0e0', zorder=0))


def my_plot(model, ax):

    ax.set_xlim(-model.p.L, model.p.L)
    ax.set_ylim(-model.p.L, model.p.L)
    ax.set_aspect('equal')
    ax.axis('off')

    L, w, R = model.p.L, model.p.w, model.p.R

    # --- Dibujar calles rectas (entradas/salidas) ---
    ax.add_patch(plt.Rectangle((-w, -L), 2*w, 2*L, color='lightgray', zorder=0))  # eje vertical
    ax.add_patch(plt.Rectangle((-L, -w), 2*L, 2*w, color='lightgray', zorder=0))  # eje horizontal

    # --- Dibujar rotonda (círculo central) ---
    circle = plt.Circle((0, 0), R, color='darkgray', fill=False, lw=3, zorder=1)
    ax.add_patch(circle)

    # --- Semáforos en las entradas ---
    lights = model.ctrl.lights()
    color_map = {
    'G': 'green',
    'R': 'red',
    'Y': 'yellow',
    'AR': 'red'   # All-Red también se muestra en rojo
    }

    locs = {
        'N1': (-w/2,  R+2),
        'S2': (+w/2, -R-2),
        'E1': ( R+2, +w/2),
        'W2': (-R-2, -w/2)
    }

    for d,(x,y) in locs.items():
        ax.add_patch(plt.Circle((x,y), 1.0, color=color_map[lights[d]], zorder=3))

    # --- Autos ---
    for c in model.cars:
        ax.add_patch(plt.Circle(c.pos, 1.0, color='blue', zorder=2))




fig, ax = plt.subplots(figsize=(6,6))
model_rotonda = FourWayModel(params_rotonda)
anim = ap.animate(model_rotonda , fig, ax, my_plot)
from IPython.display import HTML
HTML(anim.to_jshtml())
